# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My answers:
Lane: Search Intent (Lane 4)

Task type: Ranking / scoring with a classification backbone.

### Why ranking/scoring, not pure classification?

The framing skill asks: "What decision does this improve?"

My decision is "Which pages should a content editor review first for intent-content mismatch?"

That's a "which ones first?" question and the skill's table maps that directly to ranking/scoring, with a priority score as the target and precision@K as the metric.

### Why not clustering or pure classification?

- Clustering would find groups of pages but wouldn't tell me which to fix first. No action follows from a cluster alone.
- Pure classification gives a yes/no per page, but the content team can't review 30,000 pages. They need a ranked queue.
- So I use a classifier to produce a probability, then rank by that probability. The probability is the priority score.


In [3]:
#ranking/scoring framing

import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

#observed outcome we rank against
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Pages: {len(df):,}")
print(f"Declining (observed outcome): {df['is_declining_label'].mean():.1%}")

print("Task type: RANKING/SCORING")
print("Input:  page features")
print("Output: priority score (0-1), sorted descending")
print("Metric: Precision@K on the ranked list")


Pages: 30,000
Declining (observed outcome): 54.2%
Task type: RANKING/SCORING
Input:  page features
Output: priority score (0-1), sorted descending
Metric: Precision@K on the ranked list


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: "is_declining_label" (binary: 1 = declining, 0 = not declining)**
trend_direction is derived from trend_pct, which is a measured percentage change in real traffic over the trailing 90 days

So it's an observed outcome, not a defined rule.
My target is "decline" which is a proxy, not a direct intent-mismatch label.

Data gotcha observed: "trend_pct" is NaN for 'flat' and 'new' directions.
The "up" direction has a max of 44,900(an extreme outlier), consistent with the skill's warning that rate columns can be misleading. I will NOT use 'trend_pct' as a feature (leakage), so this doesn't affect my model, but it's worth noting.

In [8]:
# Building the target and verifying it's observed

import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")

# The target
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)


print("Source: trend_direction (observed from trend_pct)")
print("trend_direction distribution:")
print(df["trend_direction"].value_counts().to_string())
print("trend_pct range for each direction")
print(df.groupby("trend_direction")["trend_pct"].agg(["min", "max", "mean"]).round(2).to_string())

print("Target distribution:")
print(df["is_declining_label"].value_counts().to_string())
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")

Source: trend_direction (observed from trend_pct)
trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
trend_pct range for each direction
                   min      max    mean
trend_direction                        
down            -100.0    -20.0  -58.11
flat               NaN      NaN     NaN
new                NaN      NaN     NaN
stable           -20.0     20.0   -3.19
up                20.0  44900.0  190.67
Target distribution:
is_declining_label
1    16262
0    13738
Declining rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*


metric: Precision@50

Precision@50 = of the top 50 pages my ranking flags, what fraction are actually declining?

Why this metric:
- The action is "review the top pages", a human has limited time
- The content team can realistically review approx 50 pages per week
- Precision@K directly measures "if we act on the top K and how many were worth it"

Formula (in words):
Sort pages by my score (descending). Take the top 50. Count how many are actually declining. Divide by 50.

What number means 'good':

- Baseline (hand rule)	0.680
- Acceptable	≥ 0.75
- Good	≥ 0.85

Accuracy:
- ~54% of pages decline, so "everything declines" gets 54% accuracy — useless
- Precision@50 focuses on the top of the list, where action happens


In [5]:
# Defining and computing Precision@K on a baseline

import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values

# Baseline: hand-written rule (stale × visible)
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
baseline_score = stale * visible * df["impressions_90d"]

print("Baseline (hand rule) — Precision@K:")
for k in (20, 50, 100):
    print(f"  Precision@{k}: {precision_at_k(baseline_score, y, k):.3f}")


print("A model must beat this baseline to be useful.Target: Precision@50 ≥ 0.70")


Baseline (hand rule) — Precision@K:
  Precision@20: 0.900
  Precision@50: 0.680
  Precision@100: 0.630
A model must beat this baseline to be useful.Target: Precision@50 ≥ 0.70


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

My Answer:

one row = one content page (one pseudonymized content item)

Each row represents a single piece of content, with:
- Its content attributes (content_type, main_intent, word_count, content_age_days)
- Its search performance (impressions_90d, avg_position, ctr)
- Its observed outcome (trend_direction is my label)

The lane's slice:the columns relevant to intent-content analysis.



Observed signal: Only 2 content types exist in this dataset, and only
comparison articles pair with informational intent (697 pages).
All other content is "keyword article" across all 4 intent types.

This means intent-content mismatch is measurable but narrow, I can compare
informational × comparison article vs informational × keyword article, and
check if intent-content pairing predicts decline.

In [6]:
# Loading the lane's slice and showing the unit of analysis

import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Lane-relevant columns
lane_cols = [
    "content_id", "client_id",
    "content_type", "main_intent",                # intent-content signal
    "impressions_90d", "avg_position", "ctr",     # performance
    "word_count", "content_age_days",             # content attributes
    "trend_direction", "is_declining_label"       # observed outcome + target
]

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unit of analysis: 1 row = 1 content page")

print("MY LANE'S SLICE — first 5 pages")

display(df[lane_cols].head())


print("INTENT × CONTENT_TYPE (my core signal)")
display(df.groupby(["main_intent", "content_type"]).size().unstack(fill_value=0))


print("TARGET COLUMN (what I predict)")
display(df[["content_id", "content_type", "main_intent",
            "trend_direction", "is_declining_label"]].head(8))

Shape: 30,000 rows × 45 columns
Unit of analysis: 1 row = 1 content page
MY LANE'S SLICE — first 5 pages


,content_id,client_id,content_type,main_intent,impressions_90d,avg_position,ctr,word_count,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,10.6,0.76,3221.0,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,20.3,0.05,2481.0,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,36.5,0.09,3515.0,141,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,6.2,0.49,NaN,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,44.0,0.13,2803.0,263,down,1


INTENT × CONTENT_TYPE (my core signal)


content_type,keyword article,comparison article
main_intent,,
commercial,4612,0
informational,16538,697
navigational,46,0
transactional,5733,0


TARGET COLUMN (what I predict)


,content_id,content_type,main_intent,trend_direction,is_declining_label
0,content_304f48230142,keyword article,transactional,down,1
1,content_a1fb4e703a9e,keyword article,informational,down,1
2,content_9aa793d4d895,keyword article,informational,down,1
3,content_331d6c4de07b,keyword article,commercial,stable,0
4,content_d99b7a2d90ca,keyword article,informational,down,1
5,content_d4084a4bc775,keyword article,transactional,down,1
6,content_9a34b442b552,keyword article,informational,down,1
7,content_a63219c6e95a,keyword article,commercial,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Answer:

A learned model is better at finding declining pages than a fixed rule.
For my lane specifically, a fixed rule can't:
- Detect rare intent-content combinations (potential anomalies)
- Weigh CTR against position against age
- Learn which combinations actually correlate with decline

But a model can:
- Learn the decision boundary between aligned and misaligned
- Rank pages by risk (not just flag)
- Provide interpretable splits (depth-2 tree = readable if/else)

**Honest finding from my run:**
The hand-written rule achieves Precision@50 = 0.680,
while a depth-2 tree achieves 0.600. On MY run, the hand rule wins.

- The hand rule (stale × visible × impressions) is already a strong baseline
- A depth-2 tree is too simple, it can only ask 3 yes/no questions
- ML's advantage may appear at:
  - Deeper trees (depth 3-4)= more splits
  - Different feature sets ( engagement_rate, ai_traffic_pct)
  - Different metrics (Precision@100+ where rules run out of signal)
  - Better validation

In [7]:

# Showibg why a fixed rule fails: compare rule vs model

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Fixed rule (hand-written)
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
rule_score = stale * visible * df["impressions_90d"]

# Simple model (depth-2 tree)
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

print("Precision@50: fixed rule vs model:")
print(f"  Hand rule : {precision_at_k(rule_score, y, 50):.3f}")
print(f"  Tree      : {precision_at_k(tree_score, y, 50):.3f}")

print("The tree's readable rule:")
print(export_text(tree, feature_names=features))

Precision@50: fixed rule vs model:
  Hand rule : 0.680
  Tree      : 0.600
The tree's readable rule:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.